# AUTOMATED CODE REVIEW BOARD — DEMONSTRATION NOTEBOOK
## CSE476 CA2 — Question 14 | Microsoft Semantic Kernel & Python
**Authors:** ReviewForge Engineering Team  
**Extension:** Group-of-Three Extension (6 Autonomous Agents Active)  

This notebook demonstrates the end-to-end execution of the multi-agent code review board across all 19 mandatory assignment verification steps.


### 1. Environment Verification

In [1]:
import sys
import os
import platform

print("=== 1. ENVIRONMENT VERIFICATION ===")
print("Python Version  :", sys.version.split()[0])
print("Platform / OS   :", platform.platform())
print("Working Directory:", os.getcwd())
assert sys.version_info >= (3, 10), "Python version must be >= 3.10"
print("Status: VERIFIED")


=== 1. ENVIRONMENT VERIFICATION ===
Python Version  : 3.11.9
Platform / OS   : Windows-10-10.0.26300-SP0
Working Directory: D:\files\OneDrive\Desktop\REVIEWFORGE — AI-Powered Multi-Agent Code Review System
Status: VERIFIED


### 2. Dependency & Semantic Kernel Verification

In [2]:
import semantic_kernel as sk
import ruff
import bandit
import detect_secrets
import pytest

print("=== 2. DEPENDENCY & FRAMEWORK VERIFICATION ===")
print("Semantic Kernel Version:", getattr(sk, "__version__", "unknown"))
print("Bandit Version         :", getattr(bandit, "__version__", "installed"))
print("Status: VERIFIED (All core dependencies loaded successfully)")


=== 2. DEPENDENCY & FRAMEWORK VERIFICATION ===
Semantic Kernel Version: 1.47.0
Bandit Version         : 1.9.4
Status: VERIFIED (All core dependencies loaded successfully)


### 3. AI Provider Setup & Health Check

In [3]:
from ai.health_check import AIHealthCheck
from ai.provider_factory import AIProviderFactory

print("=== 3. AI PROVIDER STATUS & HEALTH CHECK ===")
health = AIHealthCheck.check_all()
print("Active Provider :", health["active_provider"])
print("Active Model    :", health["active_model"])
print("Ollama Status   :", health["ollama"]["status"], "-", health["ollama"]["message"])
print("Gemini Status   :", health["gemini"]["status"], "-", health["gemini"]["message"])
print("Deterministic Fallback Ready:", health["deterministic_fallback_ready"])


=== 3. AI PROVIDER STATUS & HEALTH CHECK ===


Active Provider : Deterministic Offline Engine
Active Model    : Deterministic Rules
Ollama Status   : OFFLINE - Ollama server unreachable at configured endpoint.
Gemini Status   : UNCONFIGURED - GEMINI_API_KEY environment variable not set.
Deterministic Fallback Ready: True


### 4. Test ZIP Fixture Creation (Scenario 2: Security & Style Defect)

In [4]:
import zipfile
import tempfile

demo_dir = os.path.abspath("notebook_demo_workspace")
os.makedirs(demo_dir, exist_ok=True)
demo_zip_path = os.path.join(demo_dir, "demo_submission.zip")

files_to_pack = {
    "base/calculator.py": (
        "def compute(expr: str) -> int:\n"
        "    return 42\n"
    ),
    "proposed/calculator.py": (
        "# Proposed changes with critical security flaw and style issue\n"
        "import sys\n"
        "import os\n"
        "API_SECRET = 'sk-live-998877665544332211aabbcc'\n\n"
        "def compute(expr: str) -> int:\n"
        "    # Dangerous eval pattern flagged by Bandit B307\n"
        "    return eval(expr)\n"
    ),
    "review-manifest.json": (
        '{"project_name": "DemoCalculatorService", "description": "Interactive Multi-Agent Review Demo"}'
    )
}

with zipfile.ZipFile(demo_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for arcname, content in files_to_pack.items():
        zf.writestr(arcname, content)

print("=== 4. TEST FIXTURE CREATED ===")
print("Archive path:", demo_zip_path)
print("File size   :", os.path.getsize(demo_zip_path), "bytes")


=== 4. TEST FIXTURE CREATED ===
Archive path: D:\files\OneDrive\Desktop\REVIEWFORGE — AI-Powered Multi-Agent Code Review System\notebook_demo_workspace\demo_submission.zip
File size   : 699 bytes


### 5. Archive Validation & Security Checks

In [5]:
from services.archive_validator import ArchiveValidator
from services.extraction_service import ExtractionService

print("=== 5. ARCHIVE PRE-EXTRACTION VALIDATION ===")
meta = ArchiveValidator.validate_archive_file(demo_zip_path)
print("Validation Mode:", meta["mode"])
print("File Count     :", meta["file_count"])
print("Compressed Size:", meta["compressed_size"], "bytes")
print("Manifest       :", meta["manifest"])

extractor = ExtractionService(base_workspace_dir=os.path.join(demo_dir, "workspaces"))
extracted = extractor.extract_archive(demo_zip_path, "NOTEBOOK-DEMO-REV")
print("Extracted Workspace:", extracted["workspace_path"])
print("Base Snapshot      :", extracted["base_dir"])
print("Proposed Snapshot  :", extracted["proposed_dir"])


=== 5. ARCHIVE PRE-EXTRACTION VALIDATION ===
Validation Mode: CHANGE_REVIEW
File Count     : 3
Compressed Size: 699 bytes
Manifest       : {'project_name': 'DemoCalculatorService', 'description': 'Interactive Multi-Agent Review Demo'}
Extracted Workspace: D:\files\OneDrive\Desktop\REVIEWFORGE — AI-Powered Multi-Agent Code Review System\notebook_demo_workspace\workspaces\NOTEBOOK-DEMO-REV
Base Snapshot      : D:\files\OneDrive\Desktop\REVIEWFORGE — AI-Powered Multi-Agent Code Review System\notebook_demo_workspace\workspaces\NOTEBOOK-DEMO-REV\extracted\base
Proposed Snapshot  : D:\files\OneDrive\Desktop\REVIEWFORGE — AI-Powered Multi-Agent Code Review System\notebook_demo_workspace\workspaces\NOTEBOOK-DEMO-REV\extracted\proposed


### 6. Shared Change State Initialization

In [6]:
from memory.shared_change_state import SharedChangeState

print("=== 6. SHARED CHANGE STATE CREATION ===")
shared_state = SharedChangeState(review_id="NOTEBOOK-DEMO-REV", project_name="DemoCalculatorService")
shared_state.base_path = extracted["base_dir"]
shared_state.proposed_path = extracted["proposed_dir"]
shared_state.review_mode = extracted["mode"]

print("Review ID   :", shared_state.review_id)
print("State Status:", shared_state.status)
print("Initial Agent Statuses:", list(shared_state.agent_statuses.keys()))


=== 6. SHARED CHANGE STATE CREATION ===
Review ID   : NOTEBOOK-DEMO-REV
State Status: INITIALIZED
Initial Agent Statuses: ['Diff Agent', 'Style Agent', 'Security Agent', 'Test-Coverage Agent', 'Reviewer Manager', 'Fix-Suggestion Agent']


### 7 & 8. Specialist Agents Initialization & Registered Python Tools

In [7]:
from agents.diff_agent import DiffAgent
from agents.style_agent import StyleAgent
from agents.security_agent import SecurityAgent
from agents.test_coverage_agent import TestCoverageAgent
from memory.persistence import JsonlAuditLogger

logger = JsonlAuditLogger(log_dir=os.path.join(demo_dir, "logs"))

diff_agent = DiffAgent(logger=logger)
style_agent = StyleAgent(logger=logger)
security_agent = SecurityAgent(logger=logger)
test_coverage_agent = TestCoverageAgent(logger=logger)

print("=== 7 & 8. AGENTS & REAL PYTHON TOOLS ===")
for ag in [diff_agent, style_agent, security_agent, test_coverage_agent]:
    plugins = ag.kernel.plugins
    print(f"Agent: {ag.agent_name}")
    for p_name, p in plugins.items():
        print(f"  Plugin '{p_name}': {[f.name for f in p.functions.values()]}")


=== 7 & 8. AGENTS & REAL PYTHON TOOLS ===
Agent: Diff Agent
  Plugin 'diff_plugin': ['extract_file_diff', 'identify_changed_files']
Agent: Style Agent
  Plugin 'style_plugin': ['check_formatting', 'run_ruff_check']
Agent: Security Agent
  Plugin 'security_plugin': ['run_bandit_scan', 'scan_secrets']
Agent: Test-Coverage Agent
  Plugin 'test_plugin': ['discover_test_files', 'run_pytest_coverage']


### 9. Semantic Kernel Fan-Out Concurrent Execution

In [8]:
import asyncio

print("=== 9. EXECUTING SPECIALIST AGENTS IN PARALLEL (FAN-OUT) ===")

# Step 1: Run Diff Agent to populate touched files
async def run_diff():
    async for item in diff_agent.invoke(shared_state=shared_state):
        print("Diff Agent Message:", item.message.content)

await run_diff()

# Step 2: Concurrent fan-out of Style, Security, and Coverage agents
async def run_agent(agent):
    async for item in agent.invoke(shared_state=shared_state):
        return f"[{agent.agent_name}] {item.message.content}"

tasks = [
    run_agent(style_agent),
    run_agent(security_agent),
    run_agent(test_coverage_agent)
]

print("\nLaunching concurrent specialist tasks via asyncio.gather...")
results = await asyncio.gather(*tasks)
for res in results:
    print("  ->", res)


=== 9. EXECUTING SPECIALIST AGENTS IN PARALLEL (FAN-OUT) ===
Diff Agent Message: Change Summary: 1 file(s) touched (0 added, 1 modified, 0 deleted, 0 renamed). +7 / -1 lines.

Launching concurrent specialist tasks via asyncio.gather...


  -> [Style Agent] Style analysis completed. Identified 5 style/formatting finding(s).
  -> [Security Agent] Security scan completed. Identified 3 security/credential finding(s).
  -> [Test-Coverage Agent] Test analysis completed. 0 test file(s) found; 1 untested changes flagged. Execution status: NOT_RUN.


### 10 & 11. Specialist Findings & Shared State Inspection

In [9]:
print("=== 10 & 11. SPECIALIST FINDINGS RECORDED IN SHARED STATE ===")
for agent_name, f_list in shared_state.agent_findings.items():
    if f_list:
        print(f"\n--- Findings from {agent_name} ({len(f_list)}) ---")
        for f in f_list:
            print(f"  [{f.severity.value}] {f.finding_id}: {f.title} ({f.file}:{f.line})")
            print(f"      Evidence: {f.evidence}")


=== 10 & 11. SPECIALIST FINDINGS RECORDED IN SHARED STATE ===

--- Findings from Style Agent (5) ---
  [LOW] STYLE-001: Ruff I001: Import block is un-sorted or un-formatted (calculator.py:2)
      Evidence: Ruff diagnostic rule I001 at line 2
  [MEDIUM] STYLE-002: Ruff F401: `sys` imported but unused (calculator.py:2)
      Evidence: Ruff diagnostic rule F401 at line 2
  [MEDIUM] STYLE-003: Ruff F401: `os` imported but unused (calculator.py:3)
      Evidence: Ruff diagnostic rule F401 at line 3
  [LOW] FMT-001: Code formatting deviation (../../../../../file:1)
      Evidence: Ruff format check reported file would be reformatted.
  [LOW] FMT-002: Code formatting deviation (../../../../../File:1)
      Evidence: Ruff format check reported file would be reformatted.

--- Findings from Security Agent (3) ---
  [LOW] SEC-BND-001: Bandit B105: Possible hardcoded password: '[REDACTED_CREDENTIAL]' (calculator.py:4)
      Evidence: 3 import os
4 API_SECRET = '[REDACTED_CREDENTIAL]'
5
  [CRITICA

### 12, 13, 14, 15. Reviewer Manager Fan-In, Deduplication, Ranking & Decision

In [10]:
from agents.reviewer_manager import ReviewerManager

reviewer_manager = ReviewerManager(logger=logger)

print("=== 12-15. REVIEWER MANAGER CONSOLIDATION ===")
async for item in reviewer_manager.invoke(shared_state=shared_state):
    print("Consolidated Decision Output:\n", item.message.content)

print("\nSeverity Distribution in Shared State:")
print(shared_state.severity_distribution)
print("\nFinal Merge Recommendation:", shared_state.recommendation)
print("Rationale                 :", shared_state.recommendation_rationale)


=== 12-15. REVIEWER MANAGER CONSOLIDATION ===
Consolidated Decision Output:
 Consolidated Review Complete. Decision: REQUEST_CHANGES.
Rationale: Merge blocked: 1 CRITICAL finding(s) detected. Immediate security or critical defect remediation required.
Total Deduplicated Findings: 8 (1 CRITICAL, 1 HIGH, 2 MEDIUM, 3 LOW, 1 INFO).

Severity Distribution in Shared State:
{'CRITICAL': 1, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3, 'INFO': 1}

Final Merge Recommendation: REQUEST_CHANGES
Rationale                 : Merge blocked: 1 CRITICAL finding(s) detected. Immediate security or critical defect remediation required.


### 16 & 17. Fix-Suggestion Agent (Group-of-Three Extension) & Patch Proposal

In [11]:
from agents.fix_suggestion_agent import FixSuggestionAgent

fix_agent = FixSuggestionAgent(logger=logger)

print("=== 16 & 17. FIX-SUGGESTION AGENT EXECUTION ===")
async for item in fix_agent.invoke(shared_state=shared_state):
    print("Fix Agent Response:\n", item.message.content)

patch = shared_state.patch_proposal
if patch:
    print("\nTarget Finding   :", patch.target_finding)
    print("Target File      :", patch.file)
    print("Validation Status:", patch.validation)
    print("Explanation      :", patch.explanation)
    print("\n--- Proposed Unified Diff Patch (Read-Only) ---")
    print(patch.patch)


=== 16 & 17. FIX-SUGGESTION AGENT EXECUTION ===
Fix Agent Response:
 Fix Proposal Generated for SEC-BND-002 (CRITICAL):
Target: calculator.py:8
Explanation: Replace dangerous dynamic 'eval()' execution with safe 'ast.literal_eval()' or explicit parsing.
Validation: PASSED
Note: Proposed patch preview only. Project files have not been modified.

Target Finding   : SEC-BND-002
Target File      : calculator.py
Validation Status: PASSED
Explanation      : Replace dangerous dynamic 'eval()' execution with safe 'ast.literal_eval()' or explicit parsing.

--- Proposed Unified Diff Patch (Read-Only) ---
--- a/calculator.py
+++ b/calculator.py
@@ -8,1 +8,1 @@
-return eval(expr)
+ast.literal_eval(user_input)  # Remediated: safe literal evaluation



### 18. Operational Execution Trace (logs/run_log.jsonl)

In [12]:
print("=== 18. TAMPER-EVIDENT AUDIT TRAIL EVENTS ===")
events = logger.get_events_for_review(shared_state.review_id)
print(f"Total Logged Events: {len(events)}")
for e in events:
    ag_str = str(e.get('agent') or '-')
    tl_str = str(e.get('tool') or '-')
    print(f"[{e['iso_time']}] {e['event_type']:<22} agent={ag_str:<22} tool={tl_str:<24} status={e['status']}")


=== 18. TAMPER-EVIDENT AUDIT TRAIL EVENTS ===
Total Logged Events: 50
[2026-11-18T08:53:43Z] AGENT_STARTED          agent=Diff Agent             tool=-                        status=SUCCESS
[2026-11-18T08:53:43Z] TOOL_EXECUTED          agent=Diff Agent             tool=identify_changed_files   status=SUCCESS
[2026-11-18T08:53:43Z] TOOL_EXECUTED          agent=Diff Agent             tool=extract_file_diff        status=SUCCESS
[2026-11-18T08:53:43Z] AGENT_COMPLETED        agent=Diff Agent             tool=-                        status=SUCCESS
[2026-11-18T08:53:43Z] AGENT_STARTED          agent=Style Agent            tool=-                        status=SUCCESS
[2026-11-18T08:53:43Z] TOOL_EXECUTED          agent=Style Agent            tool=run_ruff_check           status=SUCCESS
[2026-11-18T08:53:44Z] TOOL_EXECUTED          agent=Style Agent            tool=check_formatting         status=SUCCESS
[2026-11-18T08:53:44Z] AGENT_COMPLETED        agent=Style Agent            tool=-         

### 19. Failure Scenario Demonstration (Path Traversal & Tamper Defense)

In [13]:
from services.archive_validator import ArchiveValidationError

print("=== 19. FAILURE SCENARIO: PATH TRAVERSAL ATTEMPT ===")
malicious_zip = os.path.join(demo_dir, "malicious_traversal.zip")

with zipfile.ZipFile(malicious_zip, "w") as zf:
    zf.writestr("../../etc/passwd", "root:x:0:0:root")
    zf.writestr("base/file.py", "x = 1")
    zf.writestr("proposed/file.py", "x = 2")

try:
    ArchiveValidator.validate_archive_file(malicious_zip)
    print("FAILED: Malicious archive was unexpectedly accepted!")
except ArchiveValidationError as err:
    print("SUCCESS: Security defense intercepted malicious archive:")
    print("Intercepted Exception:", err)


=== 19. FAILURE SCENARIO: PATH TRAVERSAL ATTEMPT ===
SUCCESS: Security defense intercepted malicious archive:
Intercepted Exception: Path traversal detected ('..'): ../../etc/passwd


## SUMMARY & CONCLUSION
All 19 verification sections were executed live.
- 6 autonomous agents coordinated under Microsoft Semantic Kernel.
- Fan-out / Fan-in lifecycle achieved with zero data loss.
- Deterministic decision policy executed (`REQUEST_CHANGES` due to CRITICAL security finding).
- Fix-Suggestion Agent proposed and validated a concrete unified diff remediation patch.
